# Aggregate, Annotate, Normalize, and Feature Select

This notebook will run all these above operations on a single-cell file or well-aggregated file.

We will have the option to run or not 

In [ ]:
path = r"path_to_environment"

In [2]:
import pandas as pd
import numpy as np
import pycytominer
import easygui as eg
import os
import sys
import gc
sys.path.append(path)
from scripts_notebooks_fossa.profiles import generate_profiles
%load_ext autoreload
%autoreload 2

# 0) Inputs

In [153]:
def stringToBool(s):
    return s.lower() == 'yes'

profile = eg.fileopenbox(msg="Choose a file with samples and their features", default=r"D:")
print('Filename', profile)

project_name = input('Provide the name of this project:')
print('Project name:', project_name)

metadata_question = input("If you need to annotate your dataset with an external file, write yes and press enter. If already annotated, answer no and press enter.")
metadata_answer = stringToBool(metadata_question)

if metadata_answer:
    platemap = eg.fileopenbox(msg="Choose a map (csv or txt file) with plates names and metadata filenames", default=r"G:")
    platemap_path = os.path.split(platemap)[0]
    print('Platemap file selected', platemap)
    
    # Read platemap using pandas (assuming it's either CSV or TXT)
    barcode_df = pd.read_csv(platemap)
else:
    barcode_df = None  # You can assign a default value or leave it as None, depending on your needs

cells_that_run = []
cells_that_run.append(project_name)

Filename G:\Meu Drive\Doutorado Thais\Github_Analysis\Dormant_NonDormant\SampleRetrival\Dormant_x_Nondormant_SingleCells.csv
Project name: ao
Platemap file selected G:\Meu Drive\Doutorado Thais\pHD_analysisfolder\workspace\metadata\platemaps\Dormant_x_normal\metadata\barcode_platemap.csv


In [154]:
barcode_df

,Assay_Plate_Barcode,Plate_Map_Name
0,240319_144243_Plate_1_,layout_day1.txt
1,240321_153415_Plate_1_,layout_day2.txt
2,240404_134523_Plate_1_,layout_day3.txt


In [155]:
gc.collect()


11

## 1) Import extracted features file (single cell or well-aggregated)

In [156]:
df = pd.read_csv(profile)
df.head()

,Metadata_Plate,Metadata_QCFlag,Metadata_QCFlag.1,Metadata_Site,Metadata_Well,Nuclei_ImageNumber,Nuclei_ObjectNumber,Nuclei_AreaShape_Area,Nuclei_AreaShape_BoundingBoxArea,Nuclei_AreaShape_BoundingBoxMaximum_X,...,Cells_Texture_Variance_AOPI_10_02_256,Cells_Texture_Variance_AOPI_10_03_256,Cells_Texture_Variance_AOPI_20_00_256,Cells_Texture_Variance_AOPI_20_01_256,Cells_Texture_Variance_AOPI_20_02_256,Cells_Texture_Variance_AOPI_20_03_256,Cells_Texture_Variance_AOPI_5_00_256,Cells_Texture_Variance_AOPI_5_01_256,Cells_Texture_Variance_AOPI_5_02_256,Cells_Texture_Variance_AOPI_5_03_256
0,240321_153415_Plate_1_,0,0,11,D10,3,1,2609,3534,150,...,41.960698,42.253443,42.380039,44.859571,43.866346,45.269071,40.532123,40.923713,41.380130,41.272143
1,240321_153415_Plate_1_,0,0,11,D10,3,2,1837,2255,513,...,214.205638,216.504315,207.841614,207.263831,222.108959,244.118014,206.658726,209.753136,207.986281,208.201253
2,240321_153415_Plate_1_,0,0,11,D10,3,3,2466,3190,82,...,53.725273,51.841565,64.554173,59.322037,51.237739,55.302862,63.215495,60.704996,60.425950,59.798456
3,240321_153415_Plate_1_,0,0,11,D10,3,4,2002,2700,498,...,212.610204,204.039853,188.735030,163.388927,209.904213,226.798017,212.344328,215.252342,219.046176,212.632572
4,240321_153415_Plate_1_,0,0,11,D10,3,5,1835,2322,558,...,340.827507,338.439665,358.311890,364.678780,372.306522,388.147483,322.325372,339.392508,327.577464,323.398845


### 1.a) Count number of cells in each well 

In [157]:
object_count = (
                df.groupby(["Metadata_Plate", "Metadata_Well"])["Nuclei_Number_Object_Number"]
                .count()
                .reset_index()
                .rename({"Nuclei_Number_Object_Number": "Nuclei_cell_count"}, axis="columns")
            )

## 2) Generate profile

### 2A) Aggregate

- Run the next cell to list the metadata columns, and copy them to choose from which metadata information you'd like to join the rows on (strata)

- If using **already aggregated data by plates and wells**, skip to 2B.

In [158]:
pycytominer.cyto_utils.infer_cp_features(df, metadata=True)

['Metadata_Plate',
 'Metadata_QCFlag',
 'Metadata_QCFlag.1',
 'Metadata_Site',
 'Metadata_Well',
 'Metadata_Plate.1',
 'Metadata_QCFlag.2',
 'Metadata_QCFlag.1.1',
 'Metadata_Site.1',
 'Metadata_Well.1',
 'Metadata_Plate.2',
 'Metadata_QCFlag.3',
 'Metadata_QCFlag.1.2',
 'Metadata_Site.2',
 'Metadata_Well.2']

In [159]:
df1 = df

In [160]:
del df

In [161]:
gc.collect()

0

- As a default we are using **Metadata_Plate and Metadata_Well** to join the rows, using the **'median'** operation. 

In [162]:
# Supondo que df1 seja seu DataFrame original
chunksize = 1000  # Ajuste baseado na memória do seu sistema
chunk_list = []

# Dividindo o DataFrame em pedaços
for chunk in range(0, len(df1), chunksize):
    chunk_df = df1.iloc[chunk:chunk+chunksize]
    
    # Agregando cada pedaço
    aggregated_chunk = pycytominer.aggregate(
        chunk_df, 
        strata=['Metadata_Plate', 'Metadata_Well'], 
        operation='median'
    )
    
    chunk_list.append(aggregated_chunk)

# Combinando todos os pedaços agregados em um único DataFrame
df2 = pd.concat(chunk_list)

# Mostrando as primeiras linhas do DataFrame original e do DataFrame agregado
print(df1.head())
print(df2.head())

           Metadata_Plate  Metadata_QCFlag  Metadata_QCFlag.1  Metadata_Site  \
0  240321_153415_Plate_1_                0                  0             11   
1  240321_153415_Plate_1_                0                  0             11   
2  240321_153415_Plate_1_                0                  0             11   
3  240321_153415_Plate_1_                0                  0             11   
4  240321_153415_Plate_1_                0                  0             11   

  Metadata_Well  Nuclei_ImageNumber  Nuclei_ObjectNumber  \
0           D10                   3                    1   
1           D10                   3                    2   
2           D10                   3                    3   
3           D10                   3                    4   
4           D10                   3                    5   

   Nuclei_AreaShape_Area  Nuclei_AreaShape_BoundingBoxArea  \
0                   2609                              3534   
1                   1837          

### 2B) Annotate 

In [163]:
df = df2

In [164]:
del df1

In [165]:
del df2

In [166]:
gc.collect()

0

- Run the following to generate a plate list based on Metadata_Plate column.

In [167]:
plate_list = df['Metadata_Plate'].unique().tolist()

In [168]:
print(plate_list)

['240321_153415_Plate_1_', '240404_134523_Plate_1_']


In [169]:
# Supondo que df seja seu DataFrame original
df.drop_duplicates(subset=['Metadata_Plate', 'Metadata_Well'], keep='first', inplace=True)

# Exibindo as primeiras linhas do DataFrame após remover duplicados
print(df.head())


           Metadata_Plate Metadata_Well  Nuclei_ImageNumber  \
0  240321_153415_Plate_1_           D10                 5.0   
1  240321_153415_Plate_1_            D2                21.0   
2  240321_153415_Plate_1_            D3                30.0   
3  240321_153415_Plate_1_            D4                39.0   
4  240321_153415_Plate_1_            D5                55.0   

   Nuclei_ObjectNumber  Nuclei_AreaShape_Area  \
0                 15.0                 2362.0   
1                 16.0                 2401.0   
2                 12.0                 2561.0   
3                 16.5                 2613.0   
4                 12.0                 2475.0   

   Nuclei_AreaShape_BoundingBoxArea  Nuclei_AreaShape_BoundingBoxMaximum_X  \
0                            3176.0                                  736.0   
1                            3286.0                                  600.5   
2                            3432.0                                  698.0   
3             

- All metadata must be organized such:
    ```
    |- metadata 
    |   |   |- <barcode_platemap.csv> 
    |   |- platemap
    |   |   |- <platemap_1.csv>
    |   |   |- <platemap_2.csv>
    ```

- Run the next cell to annotate the profiles. 

In [170]:
df_temp_list = []
for pl in plate_list:
    df_plate = df.loc[df['Metadata_Plate'] == pl]
    #deal with metadata information
    barcode_map = barcode_df[barcode_df['Assay_Plate_Barcode'] == pl]
    index_map = barcode_map.index[0]
    metadata_filename = barcode_df['Plate_Map_Name'][index_map]
    metadata = pd.read_csv(platemap_path + r'/platemap/' + metadata_filename)
    #annotate
    df_temp = pycytominer.annotate(profiles = df_plate, platemap = metadata, join_on =  ["Metadata_well_position", "Metadata_Well"])
    df_temp_list.append(df_temp)
    print('Shape of each plate ', df_temp.shape)
df = pd.concat(df_temp_list, axis=0)

Shape of each plate  (7, 1822)
Shape of each plate  (48, 1822)


In [171]:
df

,Metadata_plate_map_name,Metadata_cell,Metadata_compound,Metadata_concentration_uM,Metadata_control_type,Metadata_Plate,Metadata_Well,Nuclei_ImageNumber,Nuclei_ObjectNumber,Nuclei_AreaShape_Area,...,Cells_Texture_Variance_AOPI_10_02_256,Cells_Texture_Variance_AOPI_10_03_256,Cells_Texture_Variance_AOPI_20_00_256,Cells_Texture_Variance_AOPI_20_01_256,Cells_Texture_Variance_AOPI_20_02_256,Cells_Texture_Variance_AOPI_20_03_256,Cells_Texture_Variance_AOPI_5_00_256,Cells_Texture_Variance_AOPI_5_01_256,Cells_Texture_Variance_AOPI_5_02_256,Cells_Texture_Variance_AOPI_5_03_256
0,layout_day2,MCF7,CoCl2,300,trt,240321_153415_Plate_1_,D2,21.0,16.0,2401.0,...,79.313272,81.060638,81.505940,83.692298,83.106544,82.273788,79.014664,76.812862,77.376023,77.292308
1,layout_day2,MCF7,CoCl2,300,trt,240321_153415_Plate_1_,D3,30.0,12.0,2561.0,...,167.159046,167.146434,176.701743,179.310752,178.924247,176.327158,165.319660,166.585682,165.616544,164.865590
2,layout_day2,MCF7,CoCl2,300,trt,240321_153415_Plate_1_,D4,39.0,16.5,2613.0,...,105.990296,106.372240,109.750079,105.306401,109.578703,111.253055,104.171554,102.921345,103.801638,103.977219
3,layout_day2,MCF7,CoCl2,300,trt,240321_153415_Plate_1_,D5,55.0,12.0,2475.0,...,149.304994,154.973374,163.633431,155.534723,158.038845,165.602048,148.172692,145.988128,148.552698,149.571166
4,layout_day2,MCF7,Non-treated,0,negcon,240321_153415_Plate_1_,D8,82.0,13.5,2547.5,...,124.827431,128.086831,130.741632,126.910012,132.165969,145.816317,123.932335,125.071066,122.922661,124.266384
5,layout_day2,MCF7,Non-treated,0,negcon,240321_153415_Plate_1_,D9,90.0,12.0,2255.0,...,126.293607,126.716892,126.199596,131.611286,127.413918,132.868554,121.934521,123.814786,122.101645,125.197131
6,layout_day2,MCF7,Non-treated,0,negcon,240321_153415_Plate_1_,D10,5.0,15.0,2362.0,...,117.172863,120.545778,122.200612,123.321553,121.337172,126.824477,116.869296,116.762572,115.867556,117.417512
0,layout_day3,MCF7,CoCl2,300,trt,240404_134523_Plate_1_,B2,28.0,24.5,2472.0,...,79.886378,81.126300,81.974851,84.407436,82.191830,84.994218,78.331306,79.250831,79.113506,80.084876
1,layout_day3,MCF7,CoCl2,300,trt,240404_134523_Plate_1_,B3,43.0,24.0,2501.0,...,107.965187,110.179120,112.727345,115.447942,114.363372,114.737525,108.003177,108.132711,106.388520,106.788609
2,layout_day3,MCF7,CoCl2,300,trt,240404_134523_Plate_1_,B4,50.0,25.0,2604.0,...,165.661592,166.878070,173.801704,177.730442,168.535731,170.314936,162.627386,164.225957,161.790491,162.922025


In [172]:
df = object_count.merge(df, on=['Metadata_Plate','Metadata_Well'])

### 2C) Normalize

- Normalize the dataset, per-plate basis, to **samples = all** or **samples = negcon**.
- CHOOSE one or the other below.

#### Normalize TO NEGCON => run next cell

In [173]:
df_temp_list = []
for pl in plate_list:
    df_temp = df.loc[df['Metadata_Plate'] == pl]
    print(df_temp.shape)
    df_norm_temp = pycytominer.normalize(df_temp, method = 'mad_robustize', mad_robustize_epsilon = 0, samples = "Metadata_control_type	 == 'negcon'") 
    df_temp_list.append(df_norm_temp)
df_norm2 = pd.concat(df_temp_list, axis=0)
cells_that_run.append('normalized_negcon')

(7, 1823)
(48, 1823)


c:\Users\Lab\miniconda3\envs\bioimage_scripts\lib\site-packages\scipy\stats\_stats_py.py:2894: RuntimeWarning: invalid value encountered in subtract
  mad = np.median(np.abs(x - med))
c:\Users\Lab\miniconda3\envs\bioimage_scripts\lib\site-packages\scipy\stats\_stats_py.py:2894: RuntimeWarning: invalid value encountered in subtract
  mad = np.median(np.abs(x - med))


In [174]:
df_norm2

,Metadata_Plate,Metadata_Well,Metadata_plate_map_name,Metadata_cell,Metadata_compound,Metadata_concentration_uM,Metadata_control_type,Nuclei_cell_count,Nuclei_ImageNumber,Nuclei_ObjectNumber,...,Cells_Texture_Variance_AOPI_10_02_256,Cells_Texture_Variance_AOPI_10_03_256,Cells_Texture_Variance_AOPI_20_00_256,Cells_Texture_Variance_AOPI_20_01_256,Cells_Texture_Variance_AOPI_20_02_256,Cells_Texture_Variance_AOPI_20_03_256,Cells_Texture_Variance_AOPI_5_00_256,Cells_Texture_Variance_AOPI_5_01_256,Cells_Texture_Variance_AOPI_5_02_256,Cells_Texture_Variance_AOPI_5_03_256
0,240321_153415_Plate_1_,D10,layout_day2,MCF7,Non-treated,0,negcon,0.000000,-6.491974,0.674491,...,-3.521361,-3.038353,-0.674491,-0.674491,-0.862514,-0.674491,-1.710093,-3.786299,-5.121499,-4.963221
1,240321_153415_Plate_1_,D2,layout_day2,MCF7,CoCl2,300,trt,0.578135,-5.142992,1.124151,...,-20.938057,-22.478892,-7.538279,-8.123250,-6.288845,-5.646139,-14.490365,-25.235105,-36.743500,-34.041037
2,240321_153415_Plate_1_,D3,layout_day2,MCF7,CoCl2,300,trt,-2.408896,-4.384190,-0.674491,...,19.473980,19.905516,8.517971,9.849302,7.311209,4.849777,14.647451,22.963487,35.748854,29.421314
3,240321_153415_Plate_1_,D4,layout_day2,MCF7,CoCl2,300,trt,15.416932,-3.625388,1.348982,...,-8.665721,-10.016705,-2.774467,-4.060639,-2.531473,-2.412189,-5.997035,-11.217587,-15.034029,-14.703093
4,240321_153415_Plate_1_,D5,layout_day2,MCF7,CoCl2,300,trt,4.336012,-2.276406,-0.674491,...,11.260510,13.912101,6.313798,5.380333,4.346803,3.652905,8.858386,11.904760,21.730369,18.337795
5,240321_153415_Plate_1_,D8,layout_day2,MCF7,Non-treated,0,negcon,-6.937619,0.000000,0.000000,...,0.000000,0.674491,0.766085,0.000000,0.674491,1.444910,0.674491,0.674491,0.674491,0.000000
6,240321_153415_Plate_1_,D9,layout_day2,MCF7,Non-treated,0,negcon,0.674491,0.674491,-0.674491,...,0.674491,0.000000,0.000000,0.883657,0.000000,0.000000,0.000000,0.000000,0.000000,0.674491
7,240404_134523_Plate_1_,B10,layout_day3,MCF7,Non-treated,0,negcon,-1.623774,-1.297415,6.070417,...,-0.282774,-0.314858,-0.186774,-0.222835,-0.283442,-0.251268,-0.272967,-0.262231,-0.257578,-0.275302
8,240404_134523_Plate_1_,B11,layout_day3,MCF7,Non-treated,0,negcon,-0.624528,-1.260287,-0.674491,...,-0.454246,-0.685884,-0.588784,-0.726937,-0.543493,-0.668782,-0.570809,-0.489828,-0.471819,-0.540509
9,240404_134523_Plate_1_,B2,layout_day3,MCF7,CoCl2,300,trt,10.891777,-1.214908,20.234723,...,-2.681498,-2.363649,-2.455418,-2.401814,-2.464043,-2.421833,-2.995991,-2.759419,-2.808894,-2.950741


### 2D) Feature selection

In [175]:
#porque o codigo abaixo estava dando erro de versao
np.bool = bool

In [176]:
df_selected = pycytominer.feature_select(df_norm2, operation = ['correlation_threshold', 'variance_threshold', 'drop_na_columns','drop_outliers'], outlier_cutoff = 500) 
print('Number of columns removed:', df_norm2.shape[1] - df_selected.shape[1])
print('Percentage of columns removed:',100 - ((df_selected.shape[1]*100)/df_norm2.shape[1]))
cells_that_run.append('feature_select')

Number of columns removed: 1400
Percentage of columns removed: 76.79648930334614


In [177]:
print(df_selected.columns.tolist())


['Metadata_Plate', 'Metadata_Well', 'Metadata_plate_map_name', 'Metadata_cell', 'Metadata_compound', 'Metadata_concentration_uM', 'Metadata_control_type', 'Nuclei_cell_count', 'Nuclei_AreaShape_CentralMoment_0_1', 'Nuclei_AreaShape_CentralMoment_1_0', 'Nuclei_AreaShape_CentralMoment_1_2', 'Nuclei_AreaShape_CentralMoment_2_1', 'Nuclei_AreaShape_EquivalentDiameter', 'Nuclei_AreaShape_Extent', 'Nuclei_AreaShape_FormFactor', 'Nuclei_AreaShape_HuMoment_2', 'Nuclei_AreaShape_HuMoment_4', 'Nuclei_AreaShape_HuMoment_5', 'Nuclei_AreaShape_HuMoment_6', 'Nuclei_AreaShape_InertiaTensor_0_0', 'Nuclei_AreaShape_InertiaTensor_1_0', 'Nuclei_AreaShape_InertiaTensor_1_1', 'Nuclei_AreaShape_MajorAxisLength', 'Nuclei_AreaShape_MaximumRadius', 'Nuclei_AreaShape_MedianRadius', 'Nuclei_AreaShape_NormalizedMoment_0_2', 'Nuclei_AreaShape_NormalizedMoment_0_3', 'Nuclei_AreaShape_NormalizedMoment_2_0', 'Nuclei_AreaShape_NormalizedMoment_2_2', 'Nuclei_AreaShape_NormalizedMoment_2_3', 'Nuclei_AreaShape_NormalizedM

In [178]:
# Encontrar colunas que contenham a palavra 'count'
count_columns = df_selected.filter(like='count').columns

print(f"Colunas que contêm a palavra 'count': {list(count_columns)}")

Colunas que contêm a palavra 'count': ['Nuclei_cell_count']


In [179]:
df_selected = df_selected.drop(columns=['Nuclei_cell_count'])

### 2E) Merge object count

# Export

In [121]:
output_path = eg.diropenbox(msg="Choose an output folder", default=r"D:")
print('Path to save the profile', output_path)

Path to save the profile G:\Meu Drive\Doutorado Thais\Github_Analysis\Dormant_NonDormant\Normalized


In [180]:
output_name = '_'.join(cells_that_run)
df_selected.to_csv(output_path + r'/' + 'dormant_x_nondormant_normalized_negcon_feature_select' + '.csv')
print('Successfully exported to:', output_path + r'/' + output_name + '.csv')

Successfully exported to: G:\Meu Drive\Doutorado Thais\Github_Analysis\Dormant_NonDormant\Normalized/ao_normalized_negcon_feature_select.csv


In [181]:
output_name = '_'.join(cells_that_run)
df_norm2.to_csv(output_path + r'/' + 'dormant_x_nondormant_normalized_negcon' + '.csv')
print('Successfully exported to:', output_path + r'/' + output_name + '.csv')

Successfully exported to: G:\Meu Drive\Doutorado Thais\Github_Analysis\Dormant_NonDormant\Normalized/ao_normalized_negcon_feature_select.csv
